In [1]:
# ============================================================
# ENTRENAMIENTO DEL MODELO - CÁNCER CERVICAL
# Dataset: SIPaKMeD
# Modelo: ResNet50 con Transfer Learning
# Autor: Danner jamanca
# ============================================================

import tensorflow as tf
print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ GPU: {tf.config.list_physical_devices('GPU')}")

✅ TensorFlow: 2.19.0
✅ GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# ============================================================
# ENTRENAMIENTO COMPLETO - CÁNCER CERVICAL
# ============================================================

import tensorflow as tf
import zipfile
import os
from google.colab import files

print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ GPU: {tf.config.list_physical_devices('GPU')}")

# 1. SUBIR Y DESCOMPRIMIR DATOS
print("\n📤 Selecciona el archivo procesado.zip")
uploaded = files.upload()

# Crear carpeta y descomprimir
os.makedirs('/content/data', exist_ok=True)
with zipfile.ZipFile('procesado.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/data')

# Verificar
print("\n📁 Verificando datos...")
DATA_PATH = '/content/data/procesado'
if os.path.exists(DATA_PATH):
    print(f"✅ Datos encontrados en {DATA_PATH}")
else:
    # Buscar la carpeta correcta
    for root, dirs, files_list in os.walk('/content/data'):
        if 'train' in dirs:
            DATA_PATH = root
            print(f"✅ Datos encontrados en {DATA_PATH}")
            break

print(f"   - Train: {len(os.listdir(f'{DATA_PATH}/train/normal')) + len(os.listdir(f'{DATA_PATH}/train/anormal'))} imágenes")
print(f"   - Validation: {len(os.listdir(f'{DATA_PATH}/validation/normal')) + len(os.listdir(f'{DATA_PATH}/validation/anormal'))} imágenes")
print(f"   - Test: {len(os.listdir(f'{DATA_PATH}/test/normal')) + len(os.listdir(f'{DATA_PATH}/test/anormal'))} imágenes")

✅ TensorFlow: 2.19.0
✅ GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

📤 Selecciona el archivo procesado.zip


Saving procesado.zip to procesado.zip

📁 Verificando datos...
✅ Datos encontrados en /content/data/procesado
   - Train: 1352 imágenes
   - Validation: 289 imágenes
   - Test: 291 imágenes


In [3]:
# ============================================================
# 2. ENTRENAR MODELO
# ============================================================

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=40, width_shift_range=0.3, height_shift_range=0.3,
    horizontal_flip=True, vertical_flip=True, zoom_range=0.3, shear_range=0.2,
    brightness_range=[0.8, 1.2], fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)

# Cargar datos
train_gen = train_datagen.flow_from_directory(f'{DATA_PATH}/train', target_size=(224,224), batch_size=16, class_mode='binary')
val_gen = val_datagen.flow_from_directory(f'{DATA_PATH}/validation', target_size=(224,224), batch_size=16, class_mode='binary')
test_gen = val_datagen.flow_from_directory(f'{DATA_PATH}/test', target_size=(224,224), batch_size=16, class_mode='binary')

# Modelo
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = True
for layer in base_model.layers[:-40]: layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer=Adam(learning_rate=0.00005), loss='binary_crossentropy', metrics=['accuracy'])

# Guardar mejor modelo automáticamente
callbacks = [
    ModelCheckpoint('modelo_cervical.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.000001, verbose=1)
]

print("\n🚀 Entrenando modelo (20-25 min)...\n")

model.fit(train_gen, epochs=60, validation_data=val_gen, callbacks=callbacks, class_weight={0:1.048, 1:0.956}, verbose=1)

# Evaluar
test_loss, test_accuracy = model.evaluate(test_gen)
print(f"\n╔════════════════════════════════════════╗")
print(f"║  ✅ PRECISIÓN FINAL: {test_accuracy*100:.2f}%             ║")
print(f"╚════════════════════════════════════════╝")

print("\n💾 Modelo guardado como: modelo_cervical.h5")

Found 1352 images belonging to 2 classes.
Found 289 images belonging to 2 classes.
Found 291 images belonging to 2 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

🚀 Entrenando modelo (20-25 min)...



/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 364ms/step - accuracy: 0.5175 - loss: 0.9600
Epoch 1: val_accuracy improved from -inf to 0.47751, saving model to modelo_cervical.h5


85/85 ━━━━━━━━━━━━━━━━━━━━ 66s 470ms/step - accuracy: 0.5178 - loss: 0.9594 - val_accuracy: 0.4775 - val_loss: 0.7390 - learning_rate: 5.0000e-05
Epoch 2/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 243ms/step - accuracy: 0.6050 - loss: 0.8111
Epoch 2: val_accuracy did not improve from 0.47751
85/85 ━━━━━━━━━━━━━━━━━━━━ 22s 254ms/step - accuracy: 0.6050 - loss: 0.8109 - val_accuracy: 0.4775 - val_loss: 0.7001 - learning_rate: 5.0000e-05
Epoch 3/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step - accuracy: 0.6055 - loss: 0.8446
Epoch 3: val_accuracy improved from 0.47751 to 0.58824, saving model to modelo_cervical.h5


85/85 ━━━━━━━━━━━━━━━━━━━━ 22s 257ms/step - accuracy: 0.6054 - loss: 0.8443 - val_accuracy: 0.5882 - val_loss: 0.6698 - learning_rate: 5.0000e-05
Epoch 4/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 240ms/step - accuracy: 0.6075 - loss: 0.8188
Epoch 4: val_accuracy did not improve from 0.58824
85/85 ━━━━━━━━━━━━━━━━━━━━ 22s 251ms/step - accuracy: 0.6075 - loss: 0.8188 - val_accuracy: 0.4775 - val_loss: 0.9611 - learning_rate: 5.0000e-05
Epoch 5/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step - accuracy: 0.6012 - loss: 0.8078
Epoch 5: val_accuracy did not improve from 0.58824
85/85 ━━━━━━━━━━━━━━━━━━━━ 21s 252ms/step - accuracy: 0.6012 - loss: 0.8079 - val_accuracy: 0.4775 - val_loss: 1.4586 - learning_rate: 5.0000e-05
Epoch 6/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step - accuracy: 0.5966 - loss: 0.8291
Epoch 6: val_accuracy improved from 0.58824 to 0.62976, saving model to modelo_cervical.h5


...

85/85 ━━━━━━━━━━━━━━━━━━━━ 22s 261ms/step - accuracy: 0.6422 - loss: 0.7660 - val_accuracy: 0.6990 - val_loss: 0.6123 - learning_rate: 5.0000e-05
Epoch 8/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step - accuracy: 0.5925 - loss: 0.8114
Epoch 8: val_accuracy did not improve from 0.69896
85/85 ━━━━━━━━━━━━━━━━━━━━ 21s 243ms/step - accuracy: 0.5929 - loss: 0.8110 - val_accuracy: 0.5952 - val_loss: 0.9647 - learning_rate: 5.0000e-05
Epoch 9/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step - accuracy: 0.6571 - loss: 0.7413
Epoch 9: val_accuracy did not improve from 0.69896
85/85 ━━━━━━━━━━━━━━━━━━━━ 22s 253ms/step - accuracy: 0.6570 - loss: 0.7412 - val_accuracy: 0.6782 - val_loss: 0.6187 - learning_rate: 5.0000e-05
Epoch 10/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 241ms/step - accuracy: 0.6796 - loss: 0.6515
Epoch 10: val_accuracy did not improve from 0.69896
85/85 ━━━━━━━━━━━━━━━━━━━━ 21s 252ms/step - accuracy: 0.6795 - loss: 0.6518 - val_accuracy: 0.5640 - val_loss: 2.3405 - learning_rate: 5.0000e-05
Epo

85/85 ━━━━━━━━━━━━━━━━━━━━ 21s 253ms/step - accuracy: 0.6815 - loss: 0.6820 - val_accuracy: 0.7370 - val_loss: 0.5288 - learning_rate: 1.0000e-05
Epoch 16/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step - accuracy: 0.6963 - loss: 0.6269
Epoch 16: val_accuracy did not improve from 0.73702
85/85 ━━━━━━━━━━━━━━━━━━━━ 22s 252ms/step - accuracy: 0.6962 - loss: 0.6271 - val_accuracy: 0.7059 - val_loss: 0.6018 - learning_rate: 1.0000e-05
Epoch 17/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 243ms/step - accuracy: 0.6747 - loss: 0.7098
Epoch 17: val_accuracy did not improve from 0.73702
85/85 ━━━━━━━━━━━━━━━━━━━━ 22s 254ms/step - accuracy: 0.6747 - loss: 0.7095 - val_accuracy: 0.7128 - val_loss: 0.5910 - learning_rate: 1.0000e-05
Epoch 18/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step - accuracy: 0.7224 - loss: 0.6175
Epoch 18: val_accuracy did not improve from 0.73702
85/85 ━━━━━━━━━━━━━━━━━━━━ 21s 253ms/step - accuracy: 0.7223 - loss: 0.6177 - val_accuracy: 0.7059 - val_loss: 0.5672 - learning_rate: 1.0000e-05

85/85 ━━━━━━━━━━━━━━━━━━━━ 22s 263ms/step - accuracy: 0.7080 - loss: 0.6167 - val_accuracy: 0.7647 - val_loss: 0.5205 - learning_rate: 1.0000e-05
Epoch 21/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 243ms/step - accuracy: 0.7183 - loss: 0.6331
Epoch 21: val_accuracy did not improve from 0.76471
85/85 ━━━━━━━━━━━━━━━━━━━━ 22s 254ms/step - accuracy: 0.7183 - loss: 0.6333 - val_accuracy: 0.7266 - val_loss: 0.5662 - learning_rate: 1.0000e-05
Epoch 22/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 243ms/step - accuracy: 0.6895 - loss: 0.6622
Epoch 22: val_accuracy did not improve from 0.76471
85/85 ━━━━━━━━━━━━━━━━━━━━ 22s 253ms/step - accuracy: 0.6898 - loss: 0.6618 - val_accuracy: 0.7439 - val_loss: 0.5337 - learning_rate: 1.0000e-05
Epoch 23/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step - accuracy: 0.6844 - loss: 0.6680
Epoch 23: val_accuracy did not improve from 0.76471
85/85 ━━━━━━━━━━━━━━━━━━━━ 21s 249ms/step - accuracy: 0.6847 - loss: 0.6675 - val_accuracy: 0.4879 - val_loss: 1.4146 - learning_rate: 1.0000e-05

85/85 ━━━━━━━━━━━━━━━━━━━━ 22s 259ms/step - accuracy: 0.6987 - loss: 0.6292 - val_accuracy: 0.7785 - val_loss: 0.4687 - learning_rate: 2.0000e-06
Epoch 28/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step - accuracy: 0.7128 - loss: 0.6017
Epoch 28: val_accuracy improved from 0.77855 to 0.79585, saving model to modelo_cervical.h5


85/85 ━━━━━━━━━━━━━━━━━━━━ 23s 266ms/step - accuracy: 0.7130 - loss: 0.6017 - val_accuracy: 0.7958 - val_loss: 0.4590 - learning_rate: 2.0000e-06
Epoch 29/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step - accuracy: 0.7183 - loss: 0.5972
Epoch 29: val_accuracy did not improve from 0.79585
85/85 ━━━━━━━━━━━━━━━━━━━━ 22s 254ms/step - accuracy: 0.7183 - loss: 0.5975 - val_accuracy: 0.7266 - val_loss: 0.5879 - learning_rate: 2.0000e-06
Epoch 30/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step - accuracy: 0.7122 - loss: 0.6446
Epoch 30: val_accuracy did not improve from 0.79585
85/85 ━━━━━━━━━━━━━━━━━━━━ 22s 264ms/step - accuracy: 0.7124 - loss: 0.6441 - val_accuracy: 0.7682 - val_loss: 0.5117 - learning_rate: 2.0000e-06
Epoch 31/60
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step - accuracy: 0.7430 - loss: 0.5768
Epoch 31: val_accuracy did not improve from 0.79585
85/85 ━━━━━━━━━━━━━━━━━━━━ 22s 259ms/step - accuracy: 0.7428 - loss: 0.5770 - val_accuracy: 0.7682 - val_loss: 0.5064 - learning_rate: 2.0000e-06

In [5]:
# ============================================================
# 3. DESCARGAR MODELO
# ============================================================

from google.colab import files
files.download('modelo_cervical.h5')
print("📥 Descargando modelo a tu PC...")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Descargando modelo a tu PC...
